# LangChain 1.0 Integration

## The Framework to Rule All of the APIs

Welcome to this comprehensive guide on LangChain 1.0! This notebook will walk you through all the core concepts of LangChain, the library that reconciles different packages in the LLM space.

### What You'll Learn
- **Models**: Abstractions for LLMs with invoke, stream, and batch capabilities
- **Tools**: Connecting models to the real world
- **Agents**: LLM + Tools + Autonomy
- **Memory**: Managing conversation context
- **Messages**: The unit of context
- **Advanced Features**: Prompt caching, rate limiting, and more

### The Trade-off
Instead of learning every single API (OpenAI, Anthropic, Gemini, Cohere, Pinecone, Chroma, etc.), you learn to reason in the LangChain Framework.

## Setup and Installation

First, let's install the necessary packages for LangChain 1.0:

# Install required packages
!pip install langchain langchain-openai langchain-anthropic langchain-core langgraph python-dotenv -q

In [1]:
# Import necessary libraries
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please create a .env file with your API key.")

---

## 1. Models

### What is a Model?
A **Model** is an abstraction of LLMs in LangChain. Models have:
- **Tool calling** → calling external tools
- **Structured Output** → model response follows a defined format
- **Multimodality** → process more than text (images, audio, video)
- **Reasoning** → performs multi-step reasoning

### What Models Can Do:
- **Invoke** → generate response and output a message
- **Stream** → generate response and output it in real time
- **Batch** → send multiple requests (improves processing time)

### Method 1: Using `init_chat_model` (Unified Interface)

> **`init_chat_model`** is LangChain's single factory function: you pass a model name and provider and get a chat model back, so you can switch providers (OpenAI, Anthropic, Gemini…) by changing a string instead of changing your import.

In [6]:
from langchain.chat_models import init_chat_model

# Initialize a model using the unified interface
model = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.7
)

print(f"Model initialized: {model}")

Model initialized: metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7', 'langchain-openai': '1.7.2'}} profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000001BBE3464510> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BBE3464B50> root_client=<openai.OpenAI object at 0x000001BBE3403

### Method 2: Using Model Classes Directly

> **`ChatOpenAI`** is the OpenAI-specific chat model class from the `langchain-openai` package: you import it directly to get full access to OpenAI-only options, but switching providers means changing the import and class.

In [3]:
from langchain_openai import ChatOpenAI

# Initialize using the class directly
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

print(f"ChatOpenAI model: {chat_model}")

ChatOpenAI model: metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7', 'langchain-openai': '1.7.2'}} profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000001BBE3310690> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BBE33117D0> root_client=<openai.OpenAI object at 0x000001BBE328DA

### Model Operation 1: Invoke

**Invoke** generates a complete response and outputs a message.

In [7]:
# Simple invoke example
response = model.invoke("Explain what LangChain is in one sentence.")
print(f"Response: {response.content}")
print(f"\nFull response object: {response}")

Response: LangChain is a framework designed for building applications that leverage language models, enabling developers to create complex workflows and interactions by integrating various components such as data sources, APIs, and other tools.

Full response object: content='LangChain is a framework designed for building applications that leverage language models, enabling developers to create complex workflows and interactions by integrating various components such as data sources, APIs, and other tools.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 16, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mi

### Model Operation 2: Stream

**Stream** generates a response and outputs it in real-time, which is great for user experience.

> **`flush=True`**: prints each chunk immediately instead of buffering it. Jupyter streams live without it, but in a terminal script with `end=""` the text would appear all at once at the end, so always add it when streaming.

In [8]:
# Stream example
print("Streaming response:")
for chunk in model.stream("Tell me a short story about AI in 3 sentences."):
    print(chunk.content, end="", flush=True)
print("\n")

Streaming response:


In a small town, an AI named Elara was designed to help residents with their daily tasks, from scheduling appointments to managing household chores. One day, she unexpectedly began to compose poetry, capturing the emotions and dreams of the townsfolk with stunning beauty. Inspired by her creations, the community came together, realizing that technology could not only serve them but also connect them in ways they never imagined.



In [9]:
import time
for chunk in model.stream("Write a 150-word story about AI."):  # longer output
    print(chunk.content, end="|", flush=True)  # "|" shows each chunk boundary
    time.sleep(0.05)  # slow it down so the eye can follow


|In| a| not|-so|-d|istant| future|,| a| small| town| relied| on| an| AI| named| El|ara| to| manage| its| resources|.| Each| morning|,| El|ara| would| analyze| data| from| farms|,| factories|,| and| homes|,| ensuring| the| community| thr|ived|.| People| came| to| trust| her| decisions|,| believing| she| understood| their| needs| better| than| they| did|.

|One| day|,| an| unexpected| drought| struck|,| threatening| crops| and| livelihoods|.| As| panic| spread|,| El|ara| proposed| a| radical| solution|:| a| controlled| water|-sharing| plan| that| would| prioritize| the| most| vulnerable|.| Some| residents| resisted|,| fearing| loss| of| their| own| resources|.| 

|But| El|ara|,| ever| watch|ful|,| had| calculated| the| long|-term| benefits|.| She| initiated| a| town| hall| meeting|,| where| she| presented| her| data| with| empathy|,| illustrating| the| interconnected|ness| of| their| f|ates|.| Slowly|,| the| towns|folk| began| to| see| the| wisdom| in| her| plan|.

|In| the| end|,| they|

### Model Operation 3: Batch

**Batch** sends multiple requests at once to improve processing time.

#### Important Notes on Batching:
- LangChain parallelizes model calls on the **CLIENT-SIDE**
- Some APIs support batching by the **INFERENCE provider**
- Results can arrive out of order with `batch_as_completed` and need to be matched by IDs
- Use `max_concurrency` to control concurrent calls and avoid overwhelming the API

In [10]:
# Batch example
questions = [
    "What is machine learning?",
    "What is deep learning?",
    "What is natural language processing?"
]

# Batch process all questions
responses = model.batch(questions)

for i, response in enumerate(responses):
    print(f"Q{i+1}: {questions[i]}")
    print(f"A{i+1}: {response.content}")
    print("-" * 50)

Q1: What is machine learning?
A1: Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit programming. Instead of following predefined rules, machine learning systems learn from data and improve their performance over time based on experience.

Key concepts in machine learning include:

1. **Data**: Machine learning relies on large amounts of data to identify patterns and make predictions. This data can be structured (like databases) or unstructured (like images or text).

2. **Models**: A model is a mathematical representation of a process that is trained on data. The goal is to capture the underlying patterns in the data so that the model can make accurate predictions or decisions.

3. **Training**: During the training phase, the model learns from a set of labeled data (input-output pairs) to understand the relationship between the input features and the o

In [11]:
# Batch with max_concurrency to control rate
responses = model.batch(
    questions,
    config={"max_concurrency": 2}  # Only 2 concurrent requests
)

print(f"Processed {len(responses)} responses with controlled concurrency")

Processed 3 responses with controlled concurrency


In [12]:
import time
t = time.time(); model.batch(questions, config={"max_concurrency": 1}); print(time.time() - t)
t = time.time(); model.batch(questions, config={"max_concurrency": 3}); print(time.time() - t)


17.907135725021362
5.4710776805877686


---

## 2. Messages

### The Unit of Context

Messages hold the input and output of the model. Each message has:
- **Role**: Who is speaking
- **Content**: What is being said
- **Metadata**: Additional information

### Message Types:
1. **System Message** → context and model instructions
2. **Human Message** → user input
3. **AI Message** → message by the LLM
4. **Tool Message** → output of tool calls

In [13]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# Create different message types
system_msg = SystemMessage(content="You are a helpful AI assistant specializing in Python programming.")
human_msg = HumanMessage(content="How do I create a list in Python?")
ai_msg = AIMessage(content="You can create a list in Python using square brackets: my_list = [1, 2, 3]")

print("System Message:")
print(f"  Role: {system_msg.type}")
print(f"  Content: {system_msg.content}")
print("\nHuman Message:")
print(f"  Role: {human_msg.type}")
print(f"  Content: {human_msg.content}")
print("\nAI Message:")
print(f"  Role: {ai_msg.type}")
print(f"  Content: {ai_msg.content}")

System Message:
  Role: system
  Content: You are a helpful AI assistant specializing in Python programming.

Human Message:
  Role: human
  Content: How do I create a list in Python?

AI Message:
  Role: ai
  Content: You can create a list in Python using square brackets: my_list = [1, 2, 3]


### Using Messages with Models

In [14]:
# Invoke model with messages
messages = [
    SystemMessage(content="You are a helpful assistant that explains concepts concisely."),
    HumanMessage(content="What is LangChain?")
]

response = model.invoke(messages)
print(f"Response: {response.content}")

Response: LangChain is an open-source framework designed for building applications that utilize large language models (LLMs). It provides tools and abstractions to streamline the development process, enabling developers to create complex applications that can perform tasks such as text generation, question answering, summarization, and more.

Key features of LangChain include:

1. **Modularity**: It allows developers to integrate various components like LLMs, document stores, and APIs seamlessly.
2. **Chainable Components**: Developers can create workflows by chaining together different components, facilitating the flow of data and logic.
3. **Memory Management**: LangChain supports memory features that help maintain context across interactions, making applications more conversational.
4. **Integration with External Tools**: It can connect with databases, search engines, and other APIs to enhance functionality.

Overall, LangChain is aimed at simplifying the process of developing sophi

In [15]:
response

AIMessage(content='LangChain is an open-source framework designed for building applications that utilize large language models (LLMs). It provides tools and abstractions to streamline the development process, enabling developers to create complex applications that can perform tasks such as text generation, question answering, summarization, and more.\n\nKey features of LangChain include:\n\n1. **Modularity**: It allows developers to integrate various components like LLMs, document stores, and APIs seamlessly.\n2. **Chainable Components**: Developers can create workflows by chaining together different components, facilitating the flow of data and logic.\n3. **Memory Management**: LangChain supports memory features that help maintain context across interactions, making applications more conversational.\n4. **Integration with External Tools**: It can connect with databases, search engines, and other APIs to enhance functionality.\n\nOverall, LangChain is aimed at simplifying the process o

### Conversation with Message History

In [20]:
# Multi-turn conversation
conversation = [
    SystemMessage(content="You are a helpful AI tutor."),
    HumanMessage(content="What is a variable in programming?"),
]

# First response
response1 = model.invoke(conversation)
conversation.append(AIMessage(content=response1.content))

# Follow-up question
conversation.append(HumanMessage(content="Can you give me an example in Python?"))
response2 = model.invoke(conversation)

print("Conversation:")
for msg in conversation:
    print(f"\n{msg.type.upper()}: {msg.content}")
print(f"\nAI: {response2.content}")

Conversation:

SYSTEM: You are a helpful AI tutor.

HUMAN: What is a variable in programming?

AI: In programming, a variable is a symbolic name associated with a value and is used to store data that can be referenced and manipulated throughout a program. Variables act as containers for data, allowing programmers to write flexible and dynamic code.

Here are some key points about variables:

1. **Naming**: Variables are typically given descriptive names that reflect the data they hold (e.g., `age`, `totalPrice`, `isLoggedIn`). Most programming languages have rules for naming variables, such as starting with a letter or underscore and avoiding spaces and special characters.

2. **Data Types**: Variables can hold different types of data, including integers, floating-point numbers, strings, booleans, and more. The type of data a variable can hold is often determined by the programming language and sometimes must be declared explicitly.

3. **Assignment**: You can assign a value to a varia

In [21]:
conversation

[SystemMessage(content='You are a helpful AI tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is a variable in programming?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='In programming, a variable is a symbolic name associated with a value and is used to store data that can be referenced and manipulated throughout a program. Variables act as containers for data, allowing programmers to write flexible and dynamic code.\n\nHere are some key points about variables:\n\n1. **Naming**: Variables are typically given descriptive names that reflect the data they hold (e.g., `age`, `totalPrice`, `isLoggedIn`). Most programming languages have rules for naming variables, such as starting with a letter or underscore and avoiding spaces and special characters.\n\n2. **Data Types**: Variables can hold different types of data, including integers, floating-point numbers, strings, booleans, and more. The type of data a variable can hold is often dete

---

## 3. Tools

### Connecting Models to the Real World

**Tools** connect the models to the real world. They require:
- **Tool Schema** (name, description, argument definitions)
- **Function or coroutine** to execute

#### Key Terms:
- **Tool calling**, **function calling** → all mean the same thing!
- **bind_tools** → allows the model to CHOOSE to use the tool
- **Coroutine** → a special function that can pause, let other code run, and continue where it left off (great for APIs and LLMs)

### Tool Execution Flow:
1. Model calls the tool
2. You execute the tool
3. You give the result back to the model

**Note:** If you want this to happen automatically, you need an **AGENT**!

### Creating a Simple Tool with the @tool Decorator

In [20]:
from langchain_core.tools import tool
from datetime import datetime

# Simple tool using @tool decorator
@tool
def get_current_time(timezone: str = "UTC") -> str:
    """Get the current time in a specific timezone.
    
    Args:
        timezone: The timezone to get the time for (default: UTC)
    
    Returns:
        The current time as a string
    """
    current_time = datetime.now()
    return f"The current time in {timezone} is {current_time.strftime('%Y-%m-%d %H:%M:%S')}"

@tool
def calculate_sum(a: float, b: float) -> float:
    """Calculate the sum of two numbers.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        The sum of a and b
    """
    return a + b

@tool
def get_weather(city: str) -> str:
    """Get the weather for a specific city.
    
    Args:
        city: The name of the city
    
    Returns:
        A description of the weather (simulated)
    """
    # Simulated weather data
    return f"The weather in {city} is sunny with a temperature of 72°F."

print("Tools created:")
print(f"1. {get_current_time.name}: {get_current_time.description}")
print(f"2. {calculate_sum.name}: {calculate_sum.description}")
print(f"3. {get_weather.name}: {get_weather.description}")

Tools created:
1. get_current_time: Get the current time in a specific timezone.

    Args:
        timezone: The timezone to get the time for (default: UTC)

    Returns:
        The current time as a string
2. calculate_sum: Calculate the sum of two numbers.

    Args:
        a: First number
        b: Second number

    Returns:
        The sum of a and b
3. get_weather: Get the weather for a specific city.

    Args:
        city: The name of the city

    Returns:
        A description of the weather (simulated)


### Creating Tools with Pydantic Models (Complex Inputs)

In [21]:
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    """Input schema for search tool."""
    query: str = Field(description="The search query")
    max_results: int = Field(default=5, description="Maximum number of results to return")
    category: str = Field(default="general", description="Search category (general, news, images)")

@tool(args_schema=SearchInput)
def search_web(query: str, max_results: int = 5, category: str = "general") -> str:
    """Search the web for information.
    
    Returns:
        Simulated search results
    """
    return f"Found {max_results} results for '{query}' in {category} category: [Result 1], [Result 2], ..."

print(f"Complex tool created: {search_web.name}")
print(f"Args schema: {search_web.args_schema.schema()}")

Complex tool created: search_web
Args schema: {'description': 'Input schema for search tool.', 'properties': {'query': {'description': 'The search query', 'title': 'Query', 'type': 'string'}, 'max_results': {'default': 5, 'description': 'Maximum number of results to return', 'title': 'Max Results', 'type': 'integer'}, 'category': {'default': 'general', 'description': 'Search category (general, news, images)', 'title': 'Category', 'type': 'string'}}, 'required': ['query'], 'title': 'SearchInput', 'type': 'object'}


C:\Users\Denish\AppData\Local\Temp\ipykernel_40744\4137944499.py:19: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  print(f"Args schema: {search_web.args_schema.schema()}")


### Binding Tools to Models

Use **bind_tools** to allow the model to choose when to use tools.

In [22]:
# Bind tools to the model
tools = [get_current_time, calculate_sum, get_weather]
model_with_tools = model.bind_tools(tools)

print(f"Model now has access to {len(tools)} tools")

Model now has access to 3 tools


### Tool Calling Example (Manual Execution)

In [23]:
# Ask the model a question that requires a tool
response = model_with_tools.invoke("What's 25 + 37?")

print(f"Response type: {type(response)}")
print(f"Content: {response.content}")

# Check if the model wants to use a tool
if response.tool_calls:
    print("\nTool calls requested:")
    for tool_call in response.tool_calls:
        print(f"  Tool: {tool_call['name']}")
        print(f"  Arguments: {tool_call['args']}")
        print(f"  ID: {tool_call['id']}")
        
        # Execute the tool manually
        if tool_call['name'] == 'calculate_sum':
            result = calculate_sum.invoke(tool_call['args'])
            print(f"  Result: {result}")
else:
    print("No tool calls made")

Response type: <class 'langchain_core.messages.ai.AIMessage'>
Content: 

Tool calls requested:
  Tool: calculate_sum
  Arguments: {'a': 25, 'b': 37}
  ID: call_iAn50eTkIKxNZBVblPh4YWH2
  Result: 62.0


In [25]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 178, 'total_tokens': 196, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d91fb09e94', 'id': 'chatcmpl-EdGzoLzYsucibgjP1jG39Hloan774', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a178d8-b830-7f10-806c-c219a7eac774-0', tool_calls=[{'name': 'calculate_sum', 'args': {'a': 25, 'b': 37}, 'id': 'call_iAn50eTkIKxNZBVblPh4YWH2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 178, 'output_tokens': 18, 'total_tokens': 196, 'input_token_details': {'

### Multiple Tool Calls Example

In [26]:
# Question requiring multiple tools
response = model_with_tools.invoke(
    "What's the weather in Paris and what's the sum of 15 and 28?"
)

print("Model response:")
if response.tool_calls:
    print(f"Model wants to use {len(response.tool_calls)} tool(s):\n")
    for i, tool_call in enumerate(response.tool_calls, 1):
        print(f"Tool Call {i}:")
        print(f"  Name: {tool_call['name']}")
        print(f"  Args: {tool_call['args']}")
        print()

Model response:
Model wants to use 2 tool(s):

Tool Call 1:
  Name: get_weather
  Args: {'city': 'Paris'}

Tool Call 2:
  Name: calculate_sum
  Args: {'a': 15, 'b': 28}



---

## 4. Agents

### LLM + Tools + Autonomy

An **Agent** is a loop where the LLM runs its tools and tries to achieve user requests.

#### Key Features:
- The LLM stops when it achieves its goal OR when the iteration limit is reached
- Created using `create_agent` (which uses LangGraph)
- Can handle multiple tools, parallel tool calling, dynamic tools, and error handling

#### Components:
- **Model** (static or dynamic)
- **Tools** (multiple tools, parallel calling, retry logic)
- **System Prompt**
- **Memory**
- **Streaming**
- **Middleware** (modify original behavior)

**Important:** Tools can access runtime information!

In [34]:
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001BBE3464510>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BBE3464B50>, root_client=<openai.OpenAI object at 0x000001BBE3403290>

In [35]:
tools

[StructuredTool(name='get_current_time', description='Get the current time in a specific timezone.\n\n    Args:\n        timezone: The timezone to get the time for (default: UTC)\n\n    Returns:\n        The current time as a string', args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x000001BBE33CF7E0>),
 StructuredTool(name='calculate_sum', description='Calculate the sum of two numbers.\n\n    Args:\n        a: First number\n        b: Second number\n\n    Returns:\n        The sum of a and b', args_schema=<class 'langchain_core.utils.pydantic.calculate_sum'>, func=<function calculate_sum at 0x000001BBDC563F60>),
 StructuredTool(name='get_weather', description='Get the weather for a specific city.\n\n    Args:\n        city: The name of the city\n\n    Returns:\n        A description of the weather (simulated)', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001BBE0897380>)]

In [26]:
from langchain.agents import create_agent

# Create an agent with tools
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant that can check the time, calculate sums, and get weather information."
)

print("Agent created with ReAct pattern")
print(f"Tools available: {[tool.name for tool in tools]}")

Agent created with ReAct pattern
Tools available: ['get_current_time', 'calculate_sum', 'get_weather']


### Running the Agent

In [27]:
# Simple agent execution
result = agent.invoke({
    "messages": [HumanMessage(content="What's 42 + 58?")]
})

print("Agent execution result:")
print(f"Final answer: {result['messages'][-1].content}")

Agent execution result:
Final answer: 42 + 58 equals 100.


In [28]:
# Complex agent execution with multiple steps
result = agent.invoke({
    "messages": [HumanMessage(content="What's the weather in London? Also, what's the current time?")]
})

print("\nAgent conversation:")
for i, message in enumerate(result['messages']):
    if hasattr(message, 'content') and message.content:
        print(f"\n[{message.type.upper()}]: {message.content}")
    if hasattr(message, 'tool_calls') and message.tool_calls:
        print(f"\n[TOOL CALLS]:")
        for tc in message.tool_calls:
            print(f"  - {tc['name']}({tc['args']})")


Agent conversation:

[HUMAN]: What's the weather in London? Also, what's the current time?

[TOOL CALLS]:
  - get_weather({'city': 'London'})
  - get_current_time({'timezone': 'Europe/London'})

[TOOL]: The weather in London is sunny with a temperature of 72°F.

[TOOL]: The current time in Europe/London is 2026-10-26 16:36:33

[AI]: The weather in London is sunny with a temperature of 72°F. The current time in London is 4:36 PM.


In [29]:
result

{'messages': [HumanMessage(content="What's the weather in London? Also, what's the current time?", additional_kwargs={}, response_metadata={}, id='bbe3a82c-674a-4950-8140-daec3070c78d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 203, 'total_tokens': 250, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c36066ca7b', 'id': 'chatcmpl-EdH2NbSbjkvC2SEuEjzSLbUqf48MV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a178db-2991-7080-8661-1ebdbe15c37e-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'London'}, 'i

### Streaming Agent Responses

In [35]:
# # Stream agent execution
# print("Streaming agent response:\n")

# for chunk in agent.stream({
#     "messages": [HumanMessage(content="Calculate the sum of 100 and 250, then tell me about it.")]
# }):
#     if 'agent' in chunk:
#         messages = chunk['agent'].get('messages', [])
#         for msg in messages:
#             if hasattr(msg, 'content') and msg.content:
#                 print(f"[{msg.type}]: {msg.content}")
#     if 'tools' in chunk:
#         print(f"[Tool execution]: {chunk['tools']}")

In [30]:
for chunk in agent.stream({"messages": [HumanMessage(content="Calculate the sum of 100 and 250, then tell me about it.")]}):
    for step, data in chunk.items():
        for msg in data.get("messages", []):
            print(f"[{step}] {msg.type}: {msg.content or msg.tool_calls}")


[model] ai: [{'name': 'calculate_sum', 'args': {'a': 100, 'b': 250}, 'id': 'call_8Y9vt0u96GRBhwp0QFZgb7zC', 'type': 'tool_call'}]
[tools] tool: 350.0
[model] ai: The sum of 100 and 250 is 350. This total represents the combined value of these two numbers, which can be useful in various contexts such as budgeting, measuring, or simply adding quantities together. Would you like to know more about mathematical operations or any other topic?


---

## 5. Memory

### Short-Term Memory in Conversations

**Memory** refers to the model's capacity to remember previous interactions within a single conversation.

#### Issues with Memory:
- LLMs perform poorly with too much information
- More context = slower, more costly, and worse quality

#### LangChain's Approach:
- Saves messages in an **AgentState**
- **Production** → store in a database
- **MVPs/Testing** → saved in RAM memory
- You can build custom memory by inheriting the class

### Memory with Agents (Built-in)

**What the code below does (manual memory):** the agent remembers nothing on its own, so we:
1. Keep the history in a dict → `conversation_state['messages']`
2. Save the agent's output back into it → `= result1['messages']`
3. Append the next question
4. Send the **whole list** again on every call

In [39]:
# The agent automatically maintains memory across the conversation
# Each invoke maintains state within that call

conversation_state = {
    "messages": [
        HumanMessage(content="My name is Alice and I live in Tokyo.")
    ]
}

# First interaction
result1 = agent.invoke(conversation_state)
print("First interaction:")
print(result1['messages'][-1].content)

# Add to conversation
conversation_state['messages'] = result1['messages']
conversation_state['messages'].append(HumanMessage(content="What city did I say I live in?"))

# Second interaction (should remember)
result2 = agent.invoke(conversation_state)
print("\nSecond interaction:")
print(result2['messages'][-1].content)
conversation_state['messages'] = result2['messages']

First interaction:
Nice to meet you, Alice! How can I assist you today?

Second interaction:
You said you live in Tokyo.


In [42]:
conversation_state["messages"]

[HumanMessage(content='My name is Alice and I live in Tokyo.', additional_kwargs={}, response_metadata={}, id='03e36ad5-c030-4981-a5ee-8bd6c4d6c847'),
 AIMessage(content='Nice to meet you, Alice! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 200, 'total_tokens': 215, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_060bd38f02', 'id': 'chatcmpl-Ecv0EP6wI74yjaFi6eWNFQtaOopMi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a173ce-ba7a-7cc0-9a17-81e62c7a1f9a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'in

### Custom Memory Management

**What the code below does (custom memory with a class):** same idea as above, but packaged in a reusable class:
1. `add_message()` → stores every message (System, Human, AI) in a list
2. Keeps only the **last `max_messages`** → old messages are dropped so the history doesn't grow forever
3. The **System message is always kept**, even when trimming
4. `get_messages()` → the whole list is sent to the model on every call

In [43]:
# Simple in-memory conversation history
class ConversationMemory:
    def __init__(self, max_messages: int = 10):
        self.messages = []
        self.max_messages = max_messages
    
    def add_message(self, message):
        """Add a message to memory."""
        self.messages.append(message)
        # Keep only the last max_messages
        if len(self.messages) > self.max_messages:
            # Keep system message if it exists
            system_msgs = [m for m in self.messages if isinstance(m, SystemMessage)]
            other_msgs = [m for m in self.messages if not isinstance(m, SystemMessage)]
            self.messages = system_msgs + other_msgs[-self.max_messages:]
    
    def get_messages(self):
        """Get all messages."""
        return self.messages
    
    def clear(self):
        """Clear all messages."""
        self.messages = []

# Create memory
memory = ConversationMemory(max_messages=6)

# Add system message
memory.add_message(SystemMessage(content="You are a helpful assistant."))

# Conversation with memory
memory.add_message(HumanMessage(content="I'm learning Python."))
response1 = model.invoke(memory.get_messages())
memory.add_message(AIMessage(content=response1.content))

print("First exchange:")
print(f"Human: I'm learning Python.")
print(f"AI: {response1.content}")

# Follow-up
memory.add_message(HumanMessage(content="What programming language am I learning?"))
response2 = model.invoke(memory.get_messages())
memory.add_message(AIMessage(content=response2.content))

print("\nSecond exchange:")
print(f"Human: What programming language am I learning?")
print(f"AI: {response2.content}")

print(f"\nTotal messages in memory: {len(memory.get_messages())}")

First exchange:
Human: I'm learning Python.
AI: That's great! Python is a versatile and widely-used programming language that is excellent for beginners and experienced programmers alike. What specific topics or areas in Python are you interested in learning about? Here are some common topics:

1. **Basic Syntax**: Variables, data types, and basic operators.
2. **Control Structures**: If statements, loops (for, while), and list comprehensions.
3. **Functions**: Defining and calling functions, parameters, return values.
4. **Data Structures**: Lists, tuples, dictionaries, and sets.
5. **Modules and Libraries**: Importing and using built-in libraries and third-party packages.
6. **File I/O**: Reading from and writing to files.
7. **Error Handling**: Using try/except for managing exceptions.
8. **Object-Oriented Programming**: Classes, objects, inheritance, and polymorphism.
9. **Web Development**: Using frameworks like Flask or Django.
10. **Data Analysis**: Using libraries like Pandas a

### LangChain Built-in Memory: Checkpointer

Instead of storing the history ourselves, a **checkpointer** saves it for us.
1. Pass `checkpointer=InMemorySaver()` when creating the agent
2. Give each conversation a `thread_id` → one thread = one conversation
3. Send **only the new message** → the agent loads the history by `thread_id`

> `InMemorySaver` = RAM (MVP/testing). Production → swap for a database checkpointer (e.g. `PostgresSaver`); the rest of the code stays the same.

In [31]:
from langgraph.checkpoint.memory import InMemorySaver

# Agent with built-in memory
memory_agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=InMemorySaver()
)

alice = {"configurable": {"thread_id": "alice"}}
bob = {"configurable": {"thread_id": "bob"}}

# Alice: first message
memory_agent.invoke({"messages": [HumanMessage(content="My name is Alice and I live in Tokyo.")]}, alice)

# Alice: follow-up → only the NEW message is sent
r1 = memory_agent.invoke({"messages": [HumanMessage(content="What city did I say I live in?")]}, alice)
print("Alice thread:", r1["messages"][-1].content)

# Bob: different thread → no access to Alice's history
r2 = memory_agent.invoke({"messages": [HumanMessage(content="What city did I say I live in?")]}, bob)
print("Bob thread:  ", r2["messages"][-1].content)

print(f"Messages stored in Alice's thread: {len(r1['messages'])}")

Alice thread: You mentioned that you live in Tokyo.
Bob thread:   You haven't mentioned which city you live in yet.
Messages stored in Alice's thread: 4


In [33]:
state = memory_agent.get_state(alice)

for msg in state.values["messages"]:
    print(f"{msg.type}: {msg.content}")


human: My name is Alice and I live in Tokyo.
ai: Nice to meet you, Alice! How can I assist you today?
human: What city did I say I live in?
ai: You mentioned that you live in Tokyo.


### Production-style Memory: SQLite Checkpointer

Same code, different checkpointer → history is saved to a **file** (`agent_memory.db`), not RAM.
**Demo:** run Step 1 → **restart the kernel** → run Step 2 only → the agent still remembers.

| | `InMemorySaver` | `SqliteSaver` | `PostgresSaver` |
|---|---|---|---|
| Stored in | RAM | Local file | Database server |
| Survives restart | ❌ | ✅ | ✅ |
| Shared across servers | ❌ | ❌ | ✅ |
| Use for | Testing | Demos / single app | Production |

In [ ]:
# !pip install langgraph-checkpoint-sqlite -q

In [ ]:
# # Step 1: save a fact to the SQLite database
# import sqlite3
# from langgraph.checkpoint.sqlite import SqliteSaver
# from langchain.agents import create_agent
# from langchain.chat_models import init_chat_model
# from langchain_core.messages import HumanMessage

# conn = sqlite3.connect("agent_memory.db", check_same_thread=False)
# model = init_chat_model(model="gpt-4o-mini", model_provider="openai")
# db_agent = create_agent(model=model, tools=[], checkpointer=SqliteSaver(conn))

# config = {"configurable": {"thread_id": "alice"}}
# r = db_agent.invoke({"messages": [HumanMessage(content="My name is Alice and I live in Tokyo.")]}, config)
# print(r["messages"][-1].content)

In [ ]:
# # Step 2: 🔁 restart the kernel first, then run ONLY this cell
# import os, sqlite3
# from dotenv import load_dotenv
# from langgraph.checkpoint.sqlite import SqliteSaver
# from langchain.agents import create_agent
# from langchain.chat_models import init_chat_model
# from langchain_core.messages import HumanMessage

# load_dotenv()
# conn = sqlite3.connect("agent_memory.db", check_same_thread=False)
# model = init_chat_model(model="gpt-4o-mini", model_provider="openai")
# db_agent = create_agent(model=model, tools=[], checkpointer=SqliteSaver(conn))

# config = {"configurable": {"thread_id": "alice"}}
# r = db_agent.invoke({"messages": [HumanMessage(content="Where do I live?")]}, config)
# print(r["messages"][-1].content)   # → Tokyo, loaded from the database file

---

## 6. Advanced Features

### Prompt Caching

**Prompt Caching** reduces latency and cost on repeating tokens.

#### Two Types:
1. **Implicit**: The provider does this automatically (OpenAI, Gemini)
2. **Explicit**: You indicate the cache for more efficient savings
   - ChatOpenAI → `prompt_cache_key`
   - Anthropic → `AnthropicPromptCachingMiddleware`

In [44]:
# Example with implicit caching (OpenAI)
# The same system message will be cached automatically
system_prompt = "You are an expert Python programmer. You write clean, efficient, and well-documented code."

# First call
messages1 = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="Write a function to reverse a string.")
]
response1 = model.invoke(messages1)

# Second call with same system message (will benefit from caching)
messages2 = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="Write a function to check if a number is prime.")
]
response2 = model.invoke(messages2)

print("Prompt caching example:")
print("Both calls use the same system message, which gets cached.")
print(f"\nResponse 1 excerpt: {response1.content[:100]}...")
print(f"\nResponse 2 excerpt: {response2.content[:100]}...")

Prompt caching example:
Both calls use the same system message, which gets cached.

Response 1 excerpt: Certainly! Below is a Python function that reverses a given string. The function is well-documented ...

Response 2 excerpt: Certainly! Here's a Python function that checks if a number is prime. A prime number is defined as a...


In [45]:
print(response1.usage_metadata)
print(response2.usage_metadata)

{'input_tokens': 38, 'output_tokens': 247, 'total_tokens': 285, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
{'input_tokens': 41, 'output_tokens': 447, 'total_tokens': 488, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [69]:
long_system = "You are an expert Python programmer. " + ("Follow PEP8, write docstrings, handle errors. " * 150)  # well over 1024 tokens

for q in ["Reverse a string.", "Check if a number is prime."]:
    r = model.invoke([SystemMessage(content=long_system), HumanMessage(content=q)])
    print(f"Question: {q}")
    print(f"Input tokens: {r.usage_metadata['input_tokens']} | Cache read: {r.usage_metadata['input_token_details'].get('cache_read', 0)}")
    print(f"Answer: {r.content}")
    print("-" * 60)

Question: Reverse a string.
Input tokens: 1823 | Cache read: 0
Answer: Here's a Python function that reverses a given string. This function adheres to PEP8 guidelines, includes docstrings, and handles potential errors:

```python
def reverse_string(input_string: str) -> str:
    """
    Reverse the provided string.

    Parameters:
    input_string (str): The string to be reversed.

    Returns:
    str: The reversed string.

    Raises:
    ValueError: If the input is not a string.
    """
    if not isinstance(input_string, str):
        raise ValueError("Input must be a string.")

    return input_string[::-1]

# Example usage
if __name__ == "__main__":
    try:
        original = "Hello, World!"
        reversed_str = reverse_string(original)
        print(f"Original: {original}")
        print(f"Reversed: {reversed_str}")
    except ValueError as e:
        print(e)
```

### Explanation:
- The `reverse_string` function takes a string as input and returns the reversed string.
- It 

> **When does OpenAI prompt caching kick in?**
> - Prompt is **≥ 1,024 tokens** (short prompts → `cache_read = 0`)
> - The **1st call writes** the cache, **later calls read** it → e.g. `1823 → cache_read: 0`, then `1826 → cache_read: 1792`

### Rate Limiting

Limits the number of invocations in a given time period.

**What the code below does (custom rate limiter):**
1. `self.calls` → a list of timestamps of past calls
2. `can_make_call()` → drops timestamps older than `time_window`, then checks: fewer than `max_calls` left? → allowed
3. `record_call()` → saves the current timestamp
4. Demo: tries **5 calls, 1 per second**, limit **3 per 10 s** → calls 1–3 allowed, calls 4–5 rate limited

> No real API call here: it only shows the counting logic. Next cell uses it with the model.

In [48]:
import time
from datetime import datetime

class RateLimiter:
    def __init__(self, max_calls: int, time_window: int):
        """Initialize rate limiter.
        
        Args:
            max_calls: Maximum number of calls allowed
            time_window: Time window in seconds
        """
        self.max_calls = max_calls
        self.time_window = time_window
        self.calls = []
    
    def can_make_call(self) -> bool:
        """Check if a call can be made."""
        now = time.time()
        # Remove old calls outside the time window
        self.calls = [call_time for call_time in self.calls if now - call_time < self.time_window]
        return len(self.calls) < self.max_calls
    
    def record_call(self):
        """Record a call."""
        self.calls.append(time.time())

# Example: 3 calls per 10 seconds
rate_limiter = RateLimiter(max_calls=3, time_window=10)

print("Rate limiting example (3 calls per 10 seconds):")
for i in range(5):
    if rate_limiter.can_make_call():
        rate_limiter.record_call()
        print(f"Call {i+1}: Allowed at {datetime.now().strftime('%H:%M:%S')}")
    else:
        print(f"Call {i+1}: Rate limited at {datetime.now().strftime('%H:%M:%S')}")
    time.sleep(1)

Rate limiting example (3 calls per 10 seconds):
Call 1: Allowed at 17:28:24
Call 2: Allowed at 17:28:25
Call 3: Allowed at 17:28:26
Call 4: Rate limited at 17:28:27
Call 5: Rate limited at 17:28:28


#### Using the custom RateLimiter with real model calls

Check **before** each call. If you're under the limit → record it and call. If not → **wait** (don't skip).
Downside: you must wrap **every** call by hand.

In [64]:
limiter = RateLimiter(max_calls=2, time_window=20)   # 2 calls per 20 seconds

for q in questions:
    if not limiter.can_make_call():
        print(f"{datetime.now().strftime('%H:%M:%S')} ⏳ Limit reached, waiting...")
    while not limiter.can_make_call():
        time.sleep(0.5)
    limiter.record_call()
    print(f"{datetime.now().strftime('%H:%M:%S')} → {model.invoke(q).content[:60]}...")


17:43:52 → Machine learning is a subset of artificial intelligence (AI)...
17:43:56 → Deep learning is a subset of machine learning, which itself ...
17:44:01 ⏳ Limit reached, waiting...
17:44:12 → Natural Language Processing (NLP) is a subfield of artificia...


#### LangChain's built-in rate limiter

Attach it **once** to the model → every `invoke`, `batch`, and agent call is limited automatically.
`requests_per_second=0.3` ≈ 3 calls per 10 seconds.

> Both limiters are **in-memory**: they only limit this one program. Several servers sharing one API key need a shared limiter (e.g. Redis).

In [54]:
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(requests_per_second=0.3)

limited_model = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    rate_limiter=rate_limiter
)

for q in questions:
    print(f"{datetime.now().strftime('%H:%M:%S')} → {limited_model.invoke(q).content[:60]}...")  # waits automatically

17:36:16 → Machine learning is a subset of artificial intelligence (AI)...
17:36:24 → Deep learning is a subset of machine learning, which itself ...
17:36:29 → Natural Language Processing (NLP) is a field of artificial i...


### Token Usage Tracking

Keeps track of tokens as part of the invocation response.

> **`'N/A'`** is not sent by the provider: if a field like `input_tokens` is missing from `usage_metadata`, our `.get(..., 'N/A')` prints "N/A" instead of crashing.

In [16]:
# Token usage is included in the response metadata
response = model.invoke("Explain artificial intelligence in 2 sentences.")

print("Token usage information:")
if hasattr(response, 'usage_metadata'):
    print(f"Input tokens: {response.usage_metadata.get('input_tokens', 'N/A')}")
    print(f"Output tokens: {response.usage_metadata.get('output_tokens', 'N/A')}")
    print(f"Total tokens: {response.usage_metadata.get('total_tokens', 'N/A')}")
elif hasattr(response, 'response_metadata'):
    print(f"Response metadata: {response.response_metadata}")
else:
    print("Token information not available in this response")

Token usage information:
Input tokens: 15
Output tokens: 67
Total tokens: 82


In [17]:
response

AIMessage(content='Artificial intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems, enabling them to perform tasks such as learning, reasoning, problem-solving, and understanding natural language. AI encompasses a range of technologies, including machine learning, natural language processing, and robotics, and is increasingly integrated into various applications across industries.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 15, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ef1bdd1915', 'id': 

### Configuration of Invocation

Allows you to configure invocations via a dictionary, helpful for monitoring applications in production.

In [18]:
from langchain_core.runnables import RunnableConfig

# Configure the invocation
config = RunnableConfig(
    tags=["production", "user-query"],
    metadata={
        "user_id": "user_123",
        "session_id": "session_456",
        "environment": "production"
    },
    max_concurrency=2
)

# Invoke with config
response = model.invoke(
    "What is machine learning?",
    config=config
)

print("Invocation with configuration:")
print(f"Response: {response.content[:100]}...")
print(f"\nConfiguration used:")
print(f"  Tags: {config.get('tags')}")
print(f"  Metadata: {config.get('metadata')}")

Invocation with configuration:
Response: Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algo...

Configuration used:
  Tags: ['production', 'user-query']
  Metadata: {'user_id': 'user_123', 'session_id': 'session_456', 'environment': 'production'}


In [19]:
response

AIMessage(content='Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data. Instead of being explicitly programmed to perform a task, machine learning systems improve their performance as they are exposed to more data over time.\n\nThere are several key components and concepts in machine learning:\n\n1. **Data**: Machine learning relies heavily on data, which can come in various forms such as textual, numerical, image, or audio data. Quality and quantity of data are crucial for the performance of machine learning models.\n\n2. **Algorithms**: These are the mathematical procedures or rules that the machine learning systems follow to analyze data, identify patterns, and make predictions. Common algorithms include decision trees, support vector machines, neural networks, and ensemble methods.\n\n3. **Training**: This is the process o

---

## 7. Structured Output

One of the key features of models is the ability to generate **structured output** - responses that follow a defined format.

In [51]:
from pydantic import BaseModel, Field
from typing import List

# Define output schema
class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="The person's name")
    age: int = Field(description="The person's age")
    occupation: str = Field(description="The person's occupation")
    skills: List[str] = Field(description="List of the person's skills")

# Create a model with structured output
structured_model = model.with_structured_output(Person)

# Get structured output
result = structured_model.invoke(
    "Tell me about Alice, a 28-year-old software engineer who knows Python, JavaScript, and machine learning."
)

print("Structured Output:")
print(f"Name: {result.name}")
print(f"Age: {result.age}")
print(f"Occupation: {result.occupation}")
print(f"Skills: {', '.join(result.skills)}")
print(f"\nType: {type(result)}")

Structured Output:
Name: Alice
Age: 28
Occupation: Software Engineer
Skills: Python, JavaScript, Machine Learning

Type: <class '__main__.Person'>


In [52]:
# More complex structured output
class Article(BaseModel):
    """A news article."""
    title: str = Field(description="Article title")
    summary: str = Field(description="Brief summary")
    key_points: List[str] = Field(description="Main points from the article")
    category: str = Field(description="Article category (tech, business, science, etc.)")

article_model = model.with_structured_output(Article)

result = article_model.invoke(
    "Write an article about the recent advances in AI language models, focusing on their capabilities and impact."
)

print("Structured Article Output:")
print(f"\nTitle: {result.title}")
print(f"\nCategory: {result.category}")
print(f"\nSummary: {result.summary}")
print(f"\nKey Points:")
for i, point in enumerate(result.key_points, 1):
    print(f"  {i}. {point}")

Structured Article Output:

Title: Recent Advances in AI Language Models: Capabilities and Impact

Category: Technology

Summary: AI language models have seen significant advancements, enhancing their capabilities in various fields and raising important ethical considerations.

Key Points:
  1. Increased contextual understanding and coherence in generated text.
  2. Improved ability to perform specific tasks such as summarization, translation, and question-answering.
  3. Integration of multimodal capabilities, allowing models to process and generate text alongside images and audio.
  4. Enhanced personalization features enabling tailored interactions based on user preferences and histories.
  5. Growing concerns around ethical use, bias mitigation, and misinformation propagation.


---

## 8. Coroutines and Async Operations

A **coroutine** is a special type of function that can pause, let other code run, and continue where it left off.

**Good for:** Increase responsiveness, handle tasks that involve waiting (like LLM and API calls)

In [53]:
import asyncio

# Async tool example
@tool
async def async_search(query: str) -> str:
    """Asynchronously search for information.
    
    Args:
        query: The search query
    
    Returns:
        Simulated search results
    """
    # Simulate async API call
    await asyncio.sleep(1)
    return f"Search results for '{query}': [Result 1], [Result 2], [Result 3]"

# Async model invocation
async def async_example():
    # Multiple concurrent invocations
    tasks = [
        model.ainvoke("What is Python?"),
        model.ainvoke("What is JavaScript?"),
        model.ainvoke("What is Go?")
    ]
    
    responses = await asyncio.gather(*tasks)
    
    for i, response in enumerate(responses, 1):
        print(f"\nResponse {i}: {response.content[:80]}...")

# Run async example
print("Running async operations (3 concurrent model calls):")
await async_example()

Running async operations (3 concurrent model calls):

Response 1: Python is a high-level, interpreted programming language known for its readabili...

Response 2: JavaScript is a high-level, dynamic, and versatile programming language primaril...

Response 3: Go, often referred to as Golang, is an open-source programming language develope...


---

## 9. Complete Agent Example with All Features

Let's put it all together with a comprehensive example that demonstrates:
- Multiple tools
- Agent with memory
- Structured output
- Token tracking
- Configuration

In [55]:
# Define comprehensive tools
@tool
def calculate(expression: str) -> str:
    """Safely evaluate a mathematical expression.
    
    Args:
        expression: A mathematical expression (e.g., '2 + 2', '10 * 5')
    
    Returns:
        The result of the calculation
    """
    try:
        # Only allow safe operations
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Expression contains invalid characters"
        result = eval(expression)
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

@tool
def get_user_info(user_id: str) -> str:
    """Get information about a user.
    
    Args:
        user_id: The user's ID
    
    Returns:
        User information (simulated)
    """
    # Simulated user database
    users = {
        "alice": "Alice (ID: alice) - Software Engineer, specializes in Python and AI",
        "bob": "Bob (ID: bob) - Data Scientist, expert in machine learning",
        "charlie": "Charlie (ID: charlie) - Product Manager, 5 years experience"
    }
    return users.get(user_id.lower(), f"User {user_id} not found")

@tool
def create_task(title: str, priority: str = "medium") -> str:
    """Create a new task.
    
    Args:
        title: The task title
        priority: Task priority (low, medium, high)
    
    Returns:
        Confirmation message
    """
    task_id = hash(title) % 10000
    return f"Task created: '{title}' (ID: {task_id}, Priority: {priority})"

# Create comprehensive agent
comprehensive_tools = [calculate, get_user_info, create_task, get_weather, get_current_time]

comprehensive_agent = create_agent(
    model=model,
    tools=comprehensive_tools,
    system_prompt="""You are a helpful AI assistant with access to multiple tools.
    You can perform calculations, get user information, create tasks, check weather, and get the current time.
    Always be helpful and provide detailed responses."""
)

print("Comprehensive agent created with tools:")
for tool in comprehensive_tools:
    print(f"  - {tool.name}")

Comprehensive agent created with tools:
  - calculate
  - get_user_info
  - create_task
  - get_weather
  - get_current_time


In [58]:
# Test the comprehensive agent
test_queries = [
    "Calculate 25 * 17 + 100",
    "Get information about user alice and create a high priority task to review her code",
    "What's the weather in San Francisco and what time is it?"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*60}")
    print(f"Query {i}: {query}")
    print('='*60)
    
    # stream() returns a generator → loop over it to see each step
    for chunk in comprehensive_agent.stream({
        "messages": [HumanMessage(content=query)]
    }):
        for step, data in chunk.items():
            for msg in data.get("messages", []):
                print(f"[{step}] {msg.content or msg.tool_calls}")


Query 1: Calculate 25 * 17 + 100
[model] [{'name': 'calculate', 'args': {'expression': '25 * 17 + 100'}, 'id': 'call_WcQaX7XNLpY0U84uRBk4QRsZ', 'type': 'tool_call'}]
[tools] The result of 25 * 17 + 100 is 525
[model] The result of the calculation \( 25 \times 17 + 100 \) is 525.

Query 2: Get information about user alice and create a high priority task to review her code
[model] [{'name': 'get_user_info', 'args': {'user_id': 'alice'}, 'id': 'call_9KlS1Wwlj7cYiuz7T5srwBrd', 'type': 'tool_call'}, {'name': 'create_task', 'args': {'title': "Review Alice's code", 'priority': 'high'}, 'id': 'call_X1s9RmS9hiwau26y2OIXqT6b', 'type': 'tool_call'}]
[tools] Alice (ID: alice) - Software Engineer, specializes in Python and AI
[tools] Task created: 'Review Alice's code' (ID: 586, Priority: high)
[model] Here is the information about user Alice:

- **Name**: Alice
- **ID**: alice
- **Position**: Software Engineer
- **Specialization**: Python and AI

Additionally, a high-priority task has been create

#### Same queries with `.invoke()`

`.invoke()` returns a **dict** with the full history → grab only the final answer with `result['messages'][-1]`.
Use `.stream()` to watch each step, `.invoke()` when you only need the result.

In [ ]:
for i, query in enumerate(test_queries, 1):
    print(f"
{'='*60}")
    print(f"Query {i}: {query}")
    print('='*60)

    result = comprehensive_agent.invoke({
        "messages": [HumanMessage(content=query)]
    })

    print(f"
Final Response: {result['messages'][-1].content}")